In [ ]:
import pandas as pd
import numpy as np

In [ ]:
cols_needed = ["order",	"species", "speciesKey", "isInvasive", "decimalLatitude", "decimalLongitude",
               "eventDate", "day", "eventTime", "elevation", "basisOfRecord", "coordinateUncertaintyInMeters", "hasCoordinate", "hasGeospatialIssues", "iucnRedListCategory"]
dtypes = {"decimalLatitude": "float32", "decimalLongitude": "float32"}

chunks = []
for chunk in pd.read_csv('/content/drive/MyDrive/occurrence.txt', sep='\t',
                          usecols=cols_needed, dtype=dtypes,
                          chunksize=200_000, low_memory=False):
    chunks.append(chunk)

df = pd.concat(chunks, ignore_index=True)
print(df.shape)

In [ ]:
df.head()

In [ ]:
df['eventDate'] = pd.to_datetime(df['eventDate'], errors='coerce').dt.date

In [ ]:
min_date = df['eventDate'].dropna().min()
max_date = df['eventDate'].dropna().max()
print(f"Minimum event date: {min_date}")
print(f"Maximum event date: {max_date}")

In [ ]:
species_counts = df['species'].value_counts()
species_to_keep = species_counts[species_counts >= 20].index
df = df[df['species'].isin(species_to_keep)]
print(len(species_to_keep))
print(df.shape)

In [ ]:
import requests
import time
import json

unique_species = df["species"].dropna().unique()
print(len(unique_species))

common_name_map = {}

for i, name in enumerate(unique_species):
    try:
        resp = requests.get(
            "https://api.inaturalist.org/v1/taxa",
            params={"q": name, "rank": "species", "per_page": 1},
            headers=headers
        )
        resp.raise_for_status()  
        results = resp.json().get("results", [])
        if results:
            common_name_map[name] = results[0].get("preferred_common_name") or name
        else:
            common_name_map[name] = name  
    except requests.exceptions.RequestException as e:
        print(f"Request failed for species '{name}': {e}")
        if resp is not None:
            print(f"Status Code: {resp.status_code}")
            print(f"Response Text: {resp.text[:500]}...") 
        common_name_map[name] = name 
    except json.JSONDecodeError as e:
        print(f"JSON Decode Error for species '{name}': {e}")
        if resp is not None:
            print(f"Status Code: {resp.status_code}")
            print(f"Response Text: {resp.text[:500]}...") 
        common_name_map[name] = name 

    time.sleep(1)  
    if (i + 1) % 100 == 0: 
        print(f" {i+1} done")


with open("common_names.json", "w") as f:
    json.dump(common_name_map, f)

df["common_name"] = df["species"].map(common_name_map)


In [ ]:
df.head()

In [ ]:
# ignore
df.to_csv('/content/drive/MyDrive/filtered_occurrence_data.csv', index=False)
print("DataFrame saved to /content/drive/MyDrive/filtered_occurrence_data.csv")

In [ ]:
df["common_name"].value_counts()


In [ ]:
import pandas as pd
import numpy as np
df = pd.read_csv(
    "../data/FIXED_filtered_occurrence_data.csv", parse_dates=["eventDate"]
)

In [ ]:
df = df.dropna(subset=["eventDate", "eventTime", "decimalLatitude", "decimalLongitude"])

df.shape

In [ ]:
import geopandas as gpd
from shapely.geometry import Point


ecoregions = gpd.read_file("../data/us_eco_l3/us_eco_l3.shp").to_crs("EPSG:4326")

points = gpd.GeoDataFrame(
    df.copy(),
    geometry=[Point(xy) for xy in zip(df["decimalLongitude"], df["decimalLatitude"])],
    crs="EPSG:4326"
)

joined = gpd.sjoin(points, ecoregions[["geometry", "US_L3NAME"]], how="left", predicate="within")
df["ecoregion"] = joined["US_L3NAME"].values

missing = df["ecoregion"].isna()
if missing.any():
    nearest = gpd.sjoin_nearest(points[missing], ecoregions[["geometry", "US_L3NAME"]])
    df.loc[missing, "ecoregion"] = nearest["US_L3NAME"].values

print(df["ecoregion"].value_counts())

In [ ]:
import rasterio
from rasterio.warp import transform

with rasterio.open("../data/Annual_NLCD_LndCov_2024_CU_C1V1/Annual_NLCD_LndCov_2024_CU_C1V1.tif") as src:
    xs, ys = transform("EPSG:4326", src.crs, df["decimalLongitude"].tolist(), df["decimalLatitude"].tolist())
    land_cover_codes = [v[0] for v in src.sample(zip(xs, ys))]

df["land_cover_code"] = land_cover_codes


nlcd_labels = {
    11: "Open Water",
    12: "Perennial Ice/Snow",
    21: "Developed, Open Space",
    22: "Developed, Low Intensity",
    23: "Developed, Medium Intensity",
    24: "Developed, High Intensity",
    31: "Barren Land",
    41: "Deciduous Forest",
    42: "Evergreen Forest",
    43: "Mixed Forest",
    51: "Dwarf Scrub",       
    52: "Shrub/Scrub",
    71: "Grassland/Herbaceous",
    72: "Sedge/Herbaceous",   
    73: "Lichens",            
    74: "Moss",               
    81: "Pasture/Hay",
    82: "Cultivated Crops",
    90: "Woody Wetlands",
    95: "Emergent Herbaceous Wetlands",
}
df["land_cover_label"] = df["land_cover_code"].map(nlcd_labels)
print(df["land_cover_label"].value_counts())

In [ ]:
df.to_csv("../data/occurrence_ecoregions.csv", index=False)

In [ ]:
import pandas as pd
import numpy as np
df = pd.read_csv("../data/occurrence_ecoregions.csv")


In [ ]:
import xarray as xr

base_path = "../data/era5/"
quarters = ["jan-march", "april-jun", "jul-sep", "oct-dec"]

merged_quarters = []

for q in quarters:
    instant_file = f"{base_path}{q}_instant.nc"
    accum_file = f"{base_path}{q}_accum.nc"

    print(f"Loading and merging {q}...")
    instant_ds = xr.open_dataset(instant_file, engine="netcdf4")
    accum_ds = xr.open_dataset(accum_file, engine="netcdf4")

    quarter_merged = xr.merge([instant_ds, accum_ds])
    merged_quarters.append(quarter_merged)

era5_full_year = xr.concat(merged_quarters, dim="valid_time")

print(era5_full_year)

In [ ]:
era5_full_year.to_netcdf("../data/era5/era5_full.nc", engine="netcdf4")
print("Saved successfully!")

In [ ]:
import xarray as xr

era5_full = xr.open_dataset("../data/era5/era5_full.nc")
print(era5_full.dims)
print(list(era5_full.data_vars))
print(era5_full.coords)
print("longitude range:", era5_full.longitude.min().item(), "to", era5_full.longitude.max().item())


In [ ]:
points = xr.Dataset({
    "latitude": ("points", df["decimalLatitude"].astype("float64").values),
    "longitude": ("points", df["decimalLongitude"].astype("float64").values),
    "valid_time": ("points", df["eventDate"].values)
})

sampled = era5_full.sel(latitude=points.latitude, longitude=points.longitude,
                         valid_time=points.valid_time, method="nearest")

df["hourly_temp_C"] = sampled["t2m"].values - 273.15
df["dewpoint_C"] = sampled["d2m"].values - 273.15
df["soil_temp_C"] = sampled["stl1"].values - 273.15
df["cloud_cover_pct"] = sampled["tcc"].values * 100
df["snow_depth_m"] = sampled["sd"].values
df["precip_m"] = sampled["tp"].values
df["precip_m"] = sampled["tp"].values

def relative_humidity(t2m_c, d2m_c):
    e = 6.112 * np.exp((17.67 * d2m_c) / (d2m_c + 243.5))
    es = 6.112 * np.exp((17.67 * t2m_c) / (t2m_c + 243.5))
    return 100 * (e / es)

df["hourly_humidity_percent"] = relative_humidity(df["hourly_temp_C"], df["dewpoint_C"])


In [ ]:
def precip_phase(row):
    if row["precip_m"] <= 0:
        return "none"
    elif row["hourly_temp_C"] < 0:
        return "snow"
    elif 0 <= row["hourly_temp_C"] < 2:
        return "mixed/freezing" 
    else:
        return "rain"

df["precip_type"] = df.apply(precip_phase, axis=1)
print(df["precip_type"].value_counts())
df.head()

In [ ]:
df.to_csv("../data/FULL_INSECT_DATA.csv", index=False)